<a href="https://colab.research.google.com/github/aceholland/learning_python/blob/main/kdsh_try_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", uploaded.keys())


Saving kdsh books.zip to kdsh books.zip
Saving kdsh_test.csv to kdsh_test.csv
Saving kdsh_train.csv to kdsh_train.csv
Uploaded files: dict_keys(['kdsh books.zip', 'kdsh_test.csv', 'kdsh_train.csv'])


In [ ]:
!pip install -q pandas scikit-learn matplotlib

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier


In [ ]:
df = pd.read_csv("kdsh_train.csv")
df.head()


,id,book_name,char,caption,content,label
0,46,In Search of the Castaways,Thalcave,NaN,Thalcave’s people faded as colonists advanced;...,consistent
1,137,The Count of Monte Cristo,Faria,The Origin of His Connection with the Count of...,"Suspected again in 1815, he was re-arrested an...",contradict
2,74,In Search of the Castaways,Kai-Koumou,NaN,Before each fight he studied the crack-pattern...,consistent
3,109,The Count of Monte Cristo,Noirtier,The Complexity of Family and Personal Life,Villefort’s drift toward the royalists disappo...,contradict
4,104,The Count of Monte Cristo,Noirtier,Involvement and Turning Point in the French Re...,His parents were targeted in a reprisal for su...,consistent


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
X = df.drop(columns=["label"])
y = df["label"]

print("Input shape:", X.shape)
print("Output shape:", y.shape)


Input shape: (80, 5)
Output shape: (80,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (64, 5)
Test size: (16, 5)


In [ ]:
X_train_processed = X_train.drop(columns=['id', 'caption', 'content'])
X_test_processed = X_test.drop(columns=['id', 'caption', 'content'])

# Apply one-hot encoding to 'book_name' and 'char'
X_train_processed = pd.get_dummies(X_train_processed, columns=['book_name', 'char'], drop_first=True)
X_test_processed = pd.get_dummies(X_test_processed, columns=['book_name', 'char'], drop_first=True)

# Align columns - this is crucial when applying get_dummies separately to train/test
train_cols = X_train_processed.columns
test_cols = X_test_processed.columns

missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    X_test_processed[c] = 0

missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    X_train_processed[c] = 0

X_test_processed = X_test_processed[train_cols] # Ensure order is the same

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train_processed, y_train)

print("Model trained ✅")

Model trained ✅


In [ ]:
y_pred = model.predict(X_test_processed)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.625
              precision    recall  f1-score   support

  consistent       0.71      0.83      0.77        12
  contradict       0.00      0.00      0.00         4

    accuracy                           0.62        16
   macro avg       0.36      0.42      0.38        16
weighted avg       0.54      0.62      0.58        16



In [ ]:
import joblib

joblib.dump(model, "model.pkl")
print("Model saved as model.pkl")


Model saved as model.pkl


In [ ]:
model = joblib.load("model.pkl")

# Example input — replace values with real ones
sample = X.iloc[[0]]

# Apply the same preprocessing to the sample data as was applied to X_train and X_test
sample_processed = sample.drop(columns=['id', 'caption', 'content'])
sample_processed = pd.get_dummies(sample_processed, columns=['book_name', 'char'], drop_first=True)

# Ensure the columns of the sample match the columns the model was trained on
# We need the `train_cols` from the GNmLcD-9_9JG cell, which is in the kernel state
# For demonstration, we'll recreate a simplified version of train_cols based on the expected output.
# In a real scenario, you'd save/load these column names or ensure the order.

# Get the columns that the model was trained on. (Assumes X_train_processed is still in memory)
train_cols = X_train_processed.columns

# Add missing columns to sample_processed and fill with 0
missing_in_sample = set(train_cols) - set(sample_processed.columns)
for c in missing_in_sample:
    sample_processed[c] = 0

# Remove extra columns from sample_processed if any (shouldn't happen often if features are aligned)
extra_in_sample = set(sample_processed.columns) - set(train_cols)
for c in extra_in_sample:
    sample_processed = sample_processed.drop(columns=[c])

# Ensure the order of columns is the same
sample_processed = sample_processed[train_cols]

prediction = model.predict(sample_processed)

print("Prediction:", prediction)

Prediction: ['consistent']


In [ ]:
output = X_test.copy()
output["true"] = y_test.values
output["predicted"] = y_pred

output.to_csv("predictions.csv", index=False)
print("Saved predictions.csv")


Saved predictions.csv
